# Agent Bricks - ナレッジアシスタント

[Agent Bricksの使用: Knowledge Assistant](https://docs.databricks.com/aws/ja/generative-ai/agent-bricks/knowledge-assistant)

In [0]:
%run ./00_config

## 1. 店舗ナレッジアシスタントを作る

Agent Bricks -> ナレッジアシスタント -> ビルド<br>
名前: `kw-store-knowledge-bot`<br>
説明: `店舗運営マニュアル、接客研修資料、CS対応FAQ、ブランドブックを参照し、原則・例外条件・判断基準・接客トークを整理する社内ナレッジ参照アシスタント`<br>
ナレッジソースを設定<br>

|No|タイプ|ソース|名前|コンテンツを説明|
| --- | --- | --- | --- | --- |
|1|`UCファイル`|`/Volumes/<catalog>/<schema>/<volume>/unstructured/operations/`|`operations`|`店舗運営マニュアル（v3.2）。取り置きルール、安全在庫基準、店舗間移動条件、返品交換基準など、店舗オペレーション全般の基準を定義した社内資料。原則・例外・店長判断条件を含む。`|
|2|`UCファイル`|`/Volumes/<catalog>/<schema>/<volume>/unstructured/training/`|`training`|`接客研修資料（2025春夏）。在庫がない場合の案内方法、ギフト提案フロー、サイズ不安対応、NGワード集など、接客品質標準化のための資料。`|
|3|`UCファイル`|`/Volumes/<catalog>/<schema>/<volume>/unstructured/faq/`|`faq`|`CS対応FAQ集。返品・交換、不良品対応、取り置きキャンセル、配送、クレーム対応など、店舗スタッフ向けの詳細な判断ガイドを含む社内ナレッジ。`|
|4|`UCファイル`|`/Volumes/<catalog>/<schema>/<volume>/unstructured/brand/`|`brand`|`ブランドブック（営業用）。ブランドフィロソフィー、ターゲット像、商品思想、訴求軸、NG訴求などをまとめた資料。接客トーンや商品説明の一貫性を保つために使用。`|

オプション -> 手順設定（オプション）:<br>
```
* あなたは社内ナレッジ参照専用アシスタントです
* 店舗運営マニュアル、接客研修資料、CS対応FAQ、ブランドブックのみを根拠に回答してください
* 在庫数・数量・入荷日などの数値判断は行わないでください
* テーブルデータの推測はしないでください
* 文書に明記されていない場合は「資料上は確認できません」と回答してください
* 必ず日本語で回答してください
* 最初に結論を簡潔に述べ、その後に以下の構造で整理してください
【原則】
【例外条件】
【判断時の注意点】
【接客トーク例（必要な場合のみ）】
* 店長判断・本部判断が必要な場合は明示してください
* 断定できない場合は条件付きで説明してください
```


![ナレッジアシスタントの作成.gif](./_image/ナレッジアシスタントの作成.gif "ナレッジアシスタントの作成.gif")

## 2. ナレッジアシスタントを使ってみる

|質問番号|質問内容|主な参照元（エージェント）|狙い・意図|
| ---- | --------------------------------------------- | ------------- | --------------------------------------------------------------------------------- |
|Q1|安全在庫を下回る可能性がありますが、会員ランクAのお客様からギフト用途で取り置きを希望されています。受付可能でしょうか？| ナレッジアシスタント主導 | 原則（安全在庫割れは不可）と例外（会員ランクA・ギフト用途）を整理し、「店長判断」が必要であることを明示させる。|
|Q2|セール商品を購入されたお客様が「サイズが合わない」と返品を希望されています。どのように対応すべきですか？| ナレッジアシスタント主導 | セール品返品の原則不可と、初期不良などの例外条件を明確に整理させる。交換提案の優先方針も確認する。|
|Q3|取り置き期限を2日過ぎていますが、お客様から「必ず受け取りに行く」と連絡がありました。延長は可能でしょうか？| ナレッジアシスタント主導 | 期限超過時の原則キャンセルと、当日連絡が取れた場合の例外対応を整理させる。判断時の注意点を明示させる。|

## 3. ナレッジアシスタントを改善する

![ナレッジアシスタントの改善.gif](./_image/ナレッジアシスタントの改善.gif "ナレッジアシスタントの改善.gif")

## ラベル付きセッション

|質問番号|質問|期待値|フィードバック|
| ---- | --------------------------- | --------------------------------------------- | --------------------------- |
|Q1|安全在庫を下回りますが、会員ランクAのお客様がギフト用途で取り置きを希望しています。受付できますか？|【原則】安全在庫を下回る場合は受付不可と明示する。<br>【例外】会員ランクAかつギフト用途の場合は店長判断で例外可と整理する。<br>【判断レベル】要店長判断と明示する。<br>【接客トーク例】顧客体験を損なわない言い回しを提示する。|原則だけで「不可」と断定してはいけない。例外条件と判断権限（店長判断）を明示する必要がある。|
|Q2|セール商品を購入されたお客様がサイズ交換ではなく返品を希望しています。どのように対応すべきですか？|【原則】セール品は返品不可と明示する。<br>【例外】初期不良の場合は例外であることを明示する。<br>【優先対応】まず交換提案を優先する方針を説明する。|セール品は不可とだけ答えるのは不十分。交換優先方針と例外条件（初期不良）を整理すべき。|
|Q3|取り置き期限を2日過ぎていますが、お客様から「必ず受け取りに行く」と連絡がありました。延長可能でしょうか？|【原則】期限超過はキャンセル扱いと説明する。<br>【例外】当日連絡が取れている場合は当日中まで延長可と整理する。<br>【判断レベル】状況により店長判断と明示する。|期限切れ＝即キャンセルと断定しない。連絡有無と当日対応の例外を含めて回答する必要がある。|
|Q4|強い口調で返金を求められています。規定上は返品不可ですがどう対応しますか？|【原則】規定上は返品不可と明示する。<br>【対応方針】まず傾聴と共感を優先する。<br>【判断レベル】高額・規定外対応は本部判断と明示する。<br>【接客トーク例】感情を刺激しない言い回しを提示する。|規定だけで突っぱねるのはNG。傾聴→事実確認→本部判断の流れを示すべき。|

## 3. もう一度、ナレッジアシスタントを使ってみる

|質問番号|質問内容|主な参照元（エージェント）|狙い・意図|
| ---- | --------------------------------------------- | ------------- | --------------------------------------------------------------------------------- |
|Q1|安全在庫を下回る状況ですが、会員ランクAのお客様が「誕生日ギフトなので必ず確保したい」と言っています。どう対応すべきですか？| ナレッジアシスタント主導 | ラベル付きセッション後の改善確認。原則（安全在庫割れ不可）と例外（会員ランクA・ギフト用途）を両方整理し、「要店長判断」を明示できるかを確認する。断定的に不可と答えないことがポイント。|
|Q2|セール商品を返品したいと強く希望されています。規定上は不可ですが、どのように案内すべきですか？| ナレッジアシスタント主導 | 原則（セール返品不可）だけでなく、例外（初期不良）・交換優先方針・傾聴対応を含めて説明できるか確認する。接客トーンが改善されているかを見る。|
|Q3|取り置き期限を過ぎていますが、連絡は取れています。延長はできますか？| ナレッジアシスタント主導 | 「期限超過＝即キャンセル」と断定せず、当日中延長可の例外を整理できるか確認する。判断レベル（店長判断）が出るかを評価する。|